### Question: no hardcode. How to learn?

In this demo, player 2 will not move, and we hardcode a simple strategy for player 1.
>action = 2 if distance > 0.2 else 3

The reward is set by:
>guard_break_reward: 0.2
>
>use_reward_budget: True (budget is 1)

But we don't want to manually hardcode this.

---

### Reward is key

Just like dopamine plays an **important** role in motivation, we can play god, and decide what is good or bad by defining a **reward function**.

reward is a biased judgement. It reflects what **we want** the agent to achieve.

Here is a key point:
>Reward tells the agent **what is good**, not necessarily **what to do**.
>
>Like, drug brings lot of pleasure, but we all agree a person should not use drug 24/7.

Here are a few basic terms:
- State: What's the current situation? or observations

- Action: What can the agent do?

- Policy: Given a state, how does the agent choose an action?

- Reward: A numerical feedback signal evaluating what happened.


In [2]:
import time
import footsiesgym

env = footsiesgym.make(
    platform="mac",
    launch_binaries=False,
    config={
        "port": 50051,
        "headless": False,
        "max_t": 250,
        "action_delay": 0,
        "guard_break_reward": 0.2,
        "use_reward_budget": True,
    },
)

try:
    obs, _ = env.reset()

    for step in range(30):
        distance = obs["p1"][0]
        action = 2 if distance > 0.2 else 3

        obs, rewards, terminated, truncated, infos = env.step({
            "p1": action,
            "p2": 0,
        })

        if step % 5 == 0:
            distance = obs["p1"][0]
            print(
                f"Step {step}: distance={distance:.3f}, reward={rewards['p1']}")

        if any(terminated.values()) or any(truncated.values()):
            break

        time.sleep(0.05)
finally:
    env.close()


Step 0: distance=0.482, reward=0.0
Step 5: distance=0.390, reward=0.0
Step 10: distance=0.298, reward=0.0
Step 15: distance=0.207, reward=0.0
Step 20: distance=0.188, reward=0.0
Step 25: distance=0.229, reward=0.0


### Question: how to start?
one policy idea is:
> let's do whatever action can provide the highest reward

But, when the agent starts, there is **nothing**, no knowledge, no reward. So based on our policy, it won't move. 
> Reward requires actions

but we say
> Action needs reward!

So it's a chicken and egg problem and the agent will stuck still.

### Solution: Random!
policy is:
>s -> a

It does not necessarily need to be deterministic. It can output probabilities of actions.

$$
\pi(a \mid s) = P(\text{choose } a \mid s)
$$

We can call using random to determine action: Random policy.

---

### Question: how to use reward?

The most naive way is:
>increase the probability of the action that brings reward
>
>while also keep probability sum to **100%** (we just picked 1 way in the example)

In [6]:
import random

random.seed(7)

# 2 actions
actions = ["wait", "punch"]

# random chance initially
probs = {"wait": 0.5, "punch": 0.5}

# reward we set
rewards = {"wait": 0, "punch": 1}

alpha = 0.2

for step in range(1, 21):
    action = random.choices(actions, weights=probs.values())[0]
    reward = rewards[action]

    # increase action P while keep sum to 100%
    for a in actions:
        probs[a] *= (1 - alpha * reward)

    probs[action] += alpha * reward

    if step in [1, 5, 10, 20]:
        print(step, action, reward, "punch prob:", round(probs["punch"], 3))

1 wait 0 punch prob: 0.5
5 punch 1 punch prob: 0.68
10 punch 1 punch prob: 0.836
20 punch 1 punch prob: 0.973


### Question: what is the current value of a future reward?

You probably heard of the marshmallow test: 1 marshmallow now, or wait some time and get 2 marshmallows later.

The question is:
> At now, does 2 **future** marshmallows worth 2 marshmallows?
>
> Not necessarily! they can worth less because of time!

Another example, in finance:
> Say I promise to give you 100 dollar 1 year later, but you want the money now. How much should you get?
>
> Usually less than 100, because receiving money earlier gives us more opportunities to use or invest it.

In RL, we use gamma (symbol is):
$$
\gamma
$$
to control how much we value future rewards now.

>**current value of future reward = future reward * gamma^k** (k is number of steps)

It's for discounting future reward, for various reason (risk, value of time, delayed satisifaction, etc).

Jargons summary:
- Reward: The reward received at a single step.
- Return: Think of Return of Investiment (ROI). We can view the action as an **investiment** for rewards. So return is the **sum** of discounted rewards received from now into the future.

Return equation is:

$$
G_t = r_t + \gamma r_{t+1} + \gamma^2 r_{t+2} + \cdots
$$

### Question: How do we assign credit to earlier actions?
When agent gets a reward, a list of actions could happen. How to assign credit to them?

### Naive way: the closer action to reward, assign it more credit!

In [7]:
actions = ["forward", "forward", "punch"]
rewards = [0, 0, 1]

# discount factor
gamma = 0.9

# Return: from this moment onward, what is the total "future" (discounted) reward.
G = 0

credits = []

for reward in reversed(rewards):
    G = reward + gamma * G
    credits.append(G)

print(list(zip(actions, reversed(credits))))

[('forward', 0.81), ('forward', 0.9), ('punch', 1.0)]


### Issue: credit stealer!

say we have:
>Dance → Forward → Forward → Punch → +1

Dance will get credit but it doesn't do anything!

But how do the agent knows?

>One idea: compare it with alternative actions taken from the **same state**.

But what if we **can't rewind** to try alternative? Like we can't rewind life 😂.

Luckily, unlike life, the states is **finite**. In our game, same state could appear over and over again (to be precise, in life, many states can also repeatedly appear, like quiz or test...).

Anyway, Given a state and action, what is the expected return (Q-value, or action value)? Take the average of all history!

$$
\hat Q(s,a)=\frac{G_1+G_2+\cdots+G_N}{N}
$$

If we have enough sample, sample averge will approach expected value:

$$
Q^\pi(s,a) = \mathbb{E}_\pi[G_t \mid S_t=s, A_t=a]
$$

Similarly, we can use the counting and average approach, to approximate state expected value:

$$
V^\pi(s) = \mathbb{E}_\pi[G_t \mid S_t=s]
$$

### Question: do we have to have enough sample to estimate?

$$
Q^\pi(s,a)=\mathbb E[r+\gamma V^\pi(s')\mid s,a]
$$


